In [259]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression

# Load datasets
health_data = pd.read_excel('health-relatedindicators.xlsx')
achievement_data = pd.read_excel('overall health achievement level.xlsx')

# Filter for selected countries
selected_countries = ['Finland', 'Italy', 'Australia', 'Canada', 'China', 
                      'Chile', 'Nigeria', 'Nepal', 'Vietnam', 'India']
filtered_health_data = health_data[health_data['Country'].isin(selected_countries)]

# Select specific indicators for health data
selected_indicators = ['sdg3_matmort', 'sdg3_neonat', 'sdg3_lifee', 'sdg3_vac']
filtered_health_data = filtered_health_data[['Country', 'year'] + selected_indicators]

# Check for missing values and handle them by forward and backward filling because they are time series
filtered_health_data = filtered_health_data.sort_values(by=['Country', 'year'])
filtered_health_data = filtered_health_data.groupby('Country').apply(lambda group: group.ffill().bfill()).reset_index(drop=True)


/var/folders/fc/m91qtdls7mld56zp96k3mm0c0000gn/T/ipykernel_70430/3982869451.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_health_data = filtered_health_data.groupby('Country').apply(lambda group: group.ffill().bfill()).reset_index(drop=True)


In [260]:
#Apply Min-Max Scaling to normalize indicators to a common scale (0 to 1), ensuring equal contribution to analysis and model training.
scaler_indicators = MinMaxScaler()
scaler_indicators.fit(filtered_health_data[selected_indicators])

# Scale health indicators in `filtered_health_data`
filtered_health_data[selected_indicators] = scaler_indicators.transform(filtered_health_data[selected_indicators])

# Scale the achievement data on 'goal3-achievementlevel' column
scaler_goal3 = MinMaxScaler()
achievement_data['goal3-achievementlevel'] = scaler_goal3.fit_transform(achievement_data[['goal3-achievementlevel']])


In [261]:
# Merge health data with achievement data
data_merged = pd.merge(
    filtered_health_data, 
    achievement_data[['Country', 'year', 'goal3-achievementlevel']], 
    on=['Country', 'year'], 
    how='inner'  
)

# Apply one-hot encoding on the 'Country' column into a numerical format
data_encoded = pd.get_dummies(data_merged, columns=['Country'], dtype=int)

# Apply one-hot encoding on the 'Country' column into a numerical format
encoder = OrdinalEncoder()
data_encoded['Year_encoded'] = encoder.fit_transform(data_encoded[['year']])

# Drop the original 'year' column to avoid redundancy
data_encoded = data_encoded.drop(columns=['year'])
data_merged.head()

,Country,year,sdg3_matmort,sdg3_neonat,sdg3_lifee,sdg3_vac,goal3-achievementlevel
0,Australia,2000,0.003275,0.048151,0.868883,0.884058,0.929859
1,Australia,2001,0.002646,0.047397,0.879195,0.898551,0.935087
2,Australia,2002,0.002297,0.046267,0.880481,0.927536,0.940058
3,Australia,2003,0.002244,0.045003,0.890954,0.927536,0.945249
4,Australia,2004,0.001904,0.043872,0.898883,0.927536,0.947878


In [262]:
# Define features (X) and target (Y) for predicting achievement level
X_step1 = data_encoded.drop(columns=['goal3-achievementlevel'])
Y_step1 = data_encoded['goal3-achievementlevel']

# Split the data for training and testing
X_train_step1, X_test_step1, Y_train_step1, Y_test_step1 = train_test_split(X_step1, Y_step1, test_size=0.2, random_state=42)

# Initialize and evaluate the model using cross-validation
model_step1 = LinearRegression()
cv_scores_step1 = cross_val_score(model_step1, X_train_step1, Y_train_step1, cv=10, scoring='r2')
print("Step 1 - Cross-validation R² scores:", cv_scores_step1)
print("Step 1 - Mean R² score:", cv_scores_step1.mean())

# Fit the model on the training data and evaluate on the test data
model_step1.fit(X_train_step1, Y_train_step1)
test_score_step1 = model_step1.score(X_test_step1, Y_test_step1)
print("Step 1 - Test R² score:", test_score_step1)


Step 1 - Cross-validation R² scores: [0.99767356 0.99840364 0.99462744 0.99545714 0.99814708 0.99784104
 0.99849715 0.99735644 0.99612495 0.99618463]
Step 1 - Mean R² score: 0.9970313059958299
Step 1 - Test R² score: 0.9980824297385585


In [263]:
# List of health indicators to predict
indicators = ['sdg3_matmort', 'sdg3_neonat', 'sdg3_lifee', 'sdg3_vac']

# Dictionary to store models for each health indicator
models_step2 = {}
for indicator in indicators:
    # Define features (X) and target (Y) for the current indicator
    X_step2 = data_encoded.drop(columns=[indicator, 'goal3-achievementlevel'])
    Y_step2 = data_encoded[indicator]
    
    # Split data into training and testing sets
    X_train_step2, X_test_step2, Y_train_step2, Y_test_step2 = train_test_split(X_step2, Y_step2, test_size=0.2, random_state=42)
    
    # Fit a Linear Regression model on the scaled data
    model = LinearRegression()
    model.fit(X_train_step2, Y_train_step2)
    
    # Evaluate the model using 10-fold cross-validation
    cv_scores_step2 = cross_val_score(model, X_train_step2, Y_train_step2, cv=10, scoring='r2')
    print(f"Step 2 - {indicator} Cross-validation R² scores:", cv_scores_step2)
    print(f"Step 2 - {indicator} Mean R² score:", cv_scores_step2.mean())
    
    # Test the model on the test data
    test_score_step2 = model.score(X_test_step2, Y_test_step2)
    print(f"Step 2 - {indicator} Test R² score:", test_score_step2)
    
    # Store the model
    models_step2[indicator] = model


Step 2 - sdg3_matmort Cross-validation R² scores: [0.99474493 0.99651713 0.99476837 0.97197092 0.99241176 0.99661935
 0.99196624 0.99457134 0.98954038 0.99381502]
Step 2 - sdg3_matmort Mean R² score: 0.9916925436005546
Step 2 - sdg3_matmort Test R² score: 0.9960772265907404
Step 2 - sdg3_neonat Cross-validation R² scores: [0.98507535 0.9837302  0.96077901 0.95796297 0.99278233 0.99098901
 0.98401565 0.97856133 0.98986234 0.98767198]
Step 2 - sdg3_neonat Mean R² score: 0.9811430157975511
Step 2 - sdg3_neonat Test R² score: 0.9920518770325015
Step 2 - sdg3_lifee Cross-validation R² scores: [0.99674651 0.99249949 0.98263497 0.98607568 0.99406852 0.99901968
 0.99511758 0.99800233 0.99587846 0.99554105]
Step 2 - sdg3_lifee Mean R² score: 0.9935584282041596
Step 2 - sdg3_lifee Test R² score: 0.9935845053435052
Step 2 - sdg3_vac Cross-validation R² scores: [0.96900717 0.93065777 0.71667319 0.64131544 0.90408211 0.94411667
 0.80603957 0.94135384 0.93034083 0.92348119]
Step 2 - sdg3_vac Mean R²

In [264]:
# Create input data for the year 2024
years = np.append(data_encoded[['Year_encoded']].values, [[2024]]).reshape(-1, 1)
encoder = OrdinalEncoder()
encoder.fit(years)
year_2024_encoded = encoder.transform([[2024]])[0][0]

# Create a DataFrame containing information for the year 2024 for each country
unique_countries = [col for col in data_encoded.columns if col.startswith('Country_')]
input_data_2024 = []
for country in unique_countries:
    row = {col: 0 for col in data_encoded.columns}
    row[country] = 1  # Set the country's column to 1 to indicate that country
    row['Year_encoded'] = year_2024_encoded  # Add the encoded year 2024
    input_data_2024.append(row)

input_df_2024 = pd.DataFrame(input_data_2024)

# Predict each health indicator for the year 2024
predicted_indicators_2024 = {}
for indicator, model in models_step2.items():
    # Remove the current indicator column from input_df_2024 to match the model
    input_features = input_df_2024.drop(columns=[indicator, 'goal3-achievementlevel']) 
    
    # Predict and store results in the dictionary
    predicted_indicators_2024[indicator] = model.predict(input_features)

# Convert prediction results into a DataFrame
predicted_indicators_2024_df = pd.DataFrame(predicted_indicators_2024)

# Print prediction results for verification
print("Prediction results for health indicators for the year 2024:")
print(predicted_indicators_2024_df)



Prediction results for health indicators for the year 2024:
   sdg3_matmort  sdg3_neonat  sdg3_lifee  sdg3_vac
0      0.003736     0.644274    0.911011 -0.114028
1      0.000877     0.642171    0.890489 -0.104412
2     -0.004359     0.638660    0.825125  0.020103
3     -0.035787     0.690502    0.751937  0.189811
4      0.014080     0.606100    0.853985 -0.036476
5     -0.090179     0.836850    0.594278  0.509278
6      0.010507     0.611402    0.909352 -0.194264
7      0.030928     0.663865    0.570147  0.523287
8      0.617088    -0.192298    0.223513  0.546836
9     -0.037205     0.684496    0.706603  0.240169


In [265]:
# Remove 'goal3-achievementlevel' and any unnecessary columns from input_df_2024
input_df_2024_for_healthlevel = input_df_2024.drop(columns=['goal3-achievementlevel', 'Predicted Achievement Level (Original Scale)'], errors='ignore')

# Predict 'goal3-achievementlevel' based on the trained model (model_step1)
predicted_health_achievement_2024 = model_step1.predict(input_df_2024_for_healthlevel)

# Inverse transform to return to the original scale
predicted_health_achievement_2024_rescaled = scaler_goal3.inverse_transform(predicted_health_achievement_2024.reshape(-1, 1))

# Add the prediction results to the DataFrame for display
input_df_2024['Predicted Achievement Level (Original Scale)'] = predicted_health_achievement_2024_rescaled

# Create a DataFrame to show country names and predicted values
# Extract country names based on the columns that start with 'Country_'
country_columns = [col for col in input_df_2024.columns if col.startswith('Country_')]

# Prepare a list to store country names and predicted achievement levels
results = []
for index, row in input_df_2024.iterrows():
    for country in country_columns:
        if row[country] == 1:  # Check if the row corresponds to the current country
            country_name = country.split('_')[1]  # Extract country name
            predicted_value = row['Predicted Achievement Level (Original Scale)']
            results.append((country_name, predicted_value))

# Create a new DataFrame for results
results_df = pd.DataFrame(results, columns=['Country', 'Predicted Achievement Level (Original Scale)'])

# Print the results DataFrame
print("Predicted achievement levels for each country in 2024:")
print(results_df)


Predicted achievement levels for each country in 2024:
     Country  Predicted Achievement Level (Original Scale)
0  Australia                                     95.353251
1     Canada                                     95.034833
2      Chile                                     88.461656
3      China                                     80.668640
4    Finland                                     94.624282
5      India                                     76.885921
6      Italy                                     92.598982
7      Nepal                                     68.948040
8    Nigeria                                     59.239523
9    Vietnam                                     75.215700


In [266]:

# Load the provided health achievement levels for 2024 from the uploaded Excel file
achievement_2024_data = pd.read_excel('HealthAcheivement-2024.xlsx')

# Clean the achievement data to extract relevant columns
achievement_2024_cleaned = achievement_2024_data[['Country', 'SDG3: Good Health and Well-Being']].copy()

# Filter the actual achievement data to only include the selected countries
selected_countries = ['Finland', 'Italy', 'Australia', 'Canada', 'China', 
                      'Chile', 'Nigeria', 'Nepal', 'Vietnam', 'India']
actual_achievement_filtered = achievement_2024_cleaned[achievement_2024_cleaned['Country'].isin(selected_countries)]

# Rename the column for clarity
actual_achievement_filtered.rename(columns={'SDG3: Good Health and Well-Being': 'Achievement Level'}, inplace=True)

# Reset index for better readability
actual_achievement_filtered.reset_index(drop=True, inplace=True)

# Print the achievement levels for the selected countries
print("Achievement Levels for Selected Countries:")
print(actual_achievement_filtered)

# Color Mapping Explanation
color_mapping = {
    'green': 'Goal Achievement',
    'yellow': 'Challenges remain',
    'orange': 'Significant challenges',
    'red': 'Major challenges',
    'grey': 'Insufficient data'
}

# Display the color mapping
print("\nColor Mapping for Achievement Levels:")
for color, description in color_mapping.items():
    print(f"{color.capitalize()}: {description}")

Achievement Levels for Selected Countries:
     Country Achievement Level
0    Finland            yellow
1      Italy            yellow
2     Canada            orange
3      Chile            yellow
4  Australia            orange
5    Vietnam               red
6      China            orange
7      Nepal               red
8      India               red
9    Nigeria               red

Color Mapping for Achievement Levels:
Green: Goal Achievement
Yellow: Challenges remain
Orange: Significant challenges
Red: Major challenges
Grey: Insufficient data


/var/folders/fc/m91qtdls7mld56zp96k3mm0c0000gn/T/ipykernel_70430/466622304.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  actual_achievement_filtered.rename(columns={'SDG3: Good Health and Well-Being': 'Achievement Level'}, inplace=True)


## Justification of Methods Utilized

1. **Handling Missing Data**: Using forward and backward filling (ffill/bfill) is essential for time series data, ensuring accurate predictions and avoiding biases in the analysis.

2. **Normalization**: MinMaxScaler standardizes the data to a uniform scale between 0 and 1, preventing negative values and enhancing the performance of machine learning models by ensuring that all features contribute equally to the prediction process.

3. **Encoding Categorical Variables**: One-hot encoding was applied to the 'Country' column to convert categorical data into a numerical format, creating binary columns for each country. Additionally, the 'year' column was encoded using ordinal encoding to represent the temporal sequence, allowing regression models to interpret the data effectively and enhancing the model's predictive power.

### Observation and Analysis of Predicted and Given Achievement Levels Comparison in 2024

**Countries with Predicted Achievement Levels Above 90%:**
- **Australia**: 95.35 - Orange (Significant challenges)
- **Canada**: 95.03 - Orange (Significant challenges)
- **Finland**: 94.62 - Yellow (Challenges remain)
- **Italy**: 92.60 - Yellow (Challenges remain)
- **Chile**: 88.46 - Orange (Significant challenges)

**Countries with Predicted Achievement Levels Below 90%:**
- **China**: 80.67 - Orange (Significant challenges)
- **Vietnam**: 75.22 - Red (Major challenges)
- **India**: 76.89 - Red (Major challenges)
- **Nepal**: 68.95 - Red (Major challenges)
- **Nigeria**: 59.24 - Red (Major challenges)

### Observations:
- **Higher Achievements (90% and above)**: Countries such as Australia, Canada, and Finland have high predicted achievement levels but still face significant challenges (orange) or remain under challenges (yellow).
  
- **Lower Achievements (Below 90%)**: Countries like China, Vietnam, India, Nepal, and Nigeria exhibit varying levels of challenges. China's predicted achievement level of 80.67% indicates significant challenges; however, it reflects a more accurate understanding of its healthcare situation compared to Vietnam and India, which have even lower predicted levels.

The comparison highlights the need for targeted interventions, particularly in lower-achieving countries. The prediction for China suggests a nuanced understanding of its healthcare challenges, emphasizing the need for reforms to enhance health outcomes.

### Prediction Accuracy Analysis
- **Model Performance**: The predictive models demonstrated strong performance, as indicated by high R² values during cross-validation, confirming their reliability in estimating health achievement levels.

- **Limitations**: Discrepancies with given achievement levels suggest that the models may not capture all influencing factors within real-world healthcare systems. The provided levels serve as benchmarks rather than actual outcomes. To improve predictive accuracy, incorporating additional factors, such as economic indicators, policy changes, and real-time health data is recommended. Regular updates to the model with the latest data can enhance its robustness and facilitate better planning in health sectors.

In summary, while the models show commendable accuracy in estimating health achievement levels, it is important to recognize that predictions may not reflect actual health outcomes. Insights from this analysis can inform strategies for improving healthcare delivery and achieving health targets in the selected countries.
